# Tutorial 01: Solving a Single Parameter Set

This tutorial demonstrates how to solve a single reaction-diffusion parameter set using the **Liaw model**,
a two-component activator-inhibitor system that models color pattern formation on ladybird (ladybug) wing covers.
The Liaw model captures how two morphogens -- an activator (u) and an inhibitor (v) -- diffuse and react on a 2D domain,
producing spatially heterogeneous concentration patterns that correspond to pigmentation.

We will walk through the full pipeline:
1. Define kinetic and diffusion parameters
2. Initialize the model with random seed points
3. Solve the PDE using an Euler forward scheme
4. Save periodic snapshots of the simulation
5. Visualize the time evolution of the pattern

## 1. Setup

Import the necessary libraries and LPF modules.

In [ ]:
import os
import os.path as osp
from os.path import join as pjoin
import time
from datetime import datetime
import json

import numpy as np
np.seterr(all='raise')
from PIL import Image

from lpf.initializers import LiawInitializer
from lpf.models import LiawModel
from lpf.solvers import EulerSolver, RungeKuttaSolver

## 2. Simulation Parameters

Define the numerical parameters for the simulation:
- **`batch_size`**: Number of parameter sets to solve (1 in this tutorial).
- **`device`**: Computation device (`"cpu"` or `"cuda:0"` for GPU).
- **`dt`**: Time step size for the Euler integration (0.01).
- **`n_iters`**: Total number of time steps (500,000 iterations = 5,000 time units).
- **`dx`**: Spatial grid spacing (0.1).
- **`width`, `height`**: Grid dimensions (128 x 128).

In [ ]:
batch_size = 1  # A single set of parameters
device = "cpu"  # Device option: CPU or GPU

# Time parameters
dt = 0.01
n_iters = 500000

# Space parameters
dx = 0.1
width = 128
height = 128
shape = (height, width)

## 3. Output Directory

Create a timestamped output directory where simulation results (images, model files) will be saved.

In [ ]:
# Create the output directory.
str_now = datetime.now().strftime('%Y%m%d-%H%M%S')
dpath_output = pjoin(osp.abspath("./output"), "experiment_batch_%s" % (str_now))
os.makedirs(dpath_output, exist_ok=True)

## 4. Kinetic Parameters

Define the kinetic parameters for the Liaw reaction-diffusion model:

| Parameter | Description |
|-----------|-------------|
| `Du`, `Dv` | Diffusion coefficients for the activator (u) and inhibitor (v). The inhibitor diffuses much faster than the activator, which is essential for Turing-type pattern formation. |
| `ru`, `rv` | Reaction rate constants controlling how strongly u and v are produced through autocatalysis. |
| `su`, `sv` | Source terms that provide a baseline production rate for each species. |
| `k` | Inhibition strength -- how strongly the inhibitor suppresses the activator. |
| `mu` | Degradation rate for both species. |
| `u0`, `v0` | Initial concentration values for the activator and inhibitor at seed points. |

In [ ]:
# Create a dict for parameters.
param_dict = {
    "u0": 2.0, "v0": 1.0,
    "Du": 0.0005, "Dv": 0.075,
    "ru": 0.18, "rv": 0.02874,
    "su": 0.001, "sv": 0.025, 
    "k": 0.084, 
    "mu": 0.08     
}

## 5. Initial Points

The activator u is initialized at specific grid positions with the value `u0`.
These seed points act as nucleation sites from which the pattern grows outward through reaction and diffusion.
Here we place 25 points at random positions on the 128x128 grid.

In [ ]:
# In this example, we use random positions for initializing u with u0.
for i in range(25):
    param_dict["init_pts_%d"%(i+1)] = (np.random.randint(0, height), np.random.randint(0, width))

In [ ]:
param_dict

The parameter dictionary must be wrapped in a list to conform to the batch API, even when solving a single model.

In [ ]:
model_dicts = []
model_dicts.append(param_dict)

## 6. Model Construction

Building the model involves three steps:
1. **`LiawInitializer`**: Reads the initial point positions from the parameter dicts and prepares the initial concentration fields.
2. **`LiawModel.parse_params`**: Extracts the kinetic parameters (Du, Dv, ru, rv, etc.) from the dicts and stacks them into arrays suitable for batch computation.
3. **`LiawModel`**: Constructs the model object with the initializer, parsed parameters, grid spacing, and device.

In [ ]:
# Create the Liaw initializer.
initializer = LiawInitializer()
initializer.update(model_dicts)
params = LiawModel.parse_params(model_dicts)

In [ ]:
# Create the Liaw model.
model = LiawModel(
    initializer=initializer,
    params=params,
    dx=dx,
    width=width,
    height=height,
    device=device
)

## 7. Solve

Run the simulation using the **Euler forward method** (explicit time-stepping).
The solver advances the PDE for `n_iters` steps of size `dt`.

- **`period_output=10000`**: Save snapshot images and model state every 10,000 iterations.
- **`dpath_model`**: Directory for saving model JSON files.
- **`dpath_morph`**: Directory for saving morph (ladybird overlay) images.
- **`dpath_pattern`**: Directory for saving raw pattern images.
- **`verbose=1`**: Print progress information during the solve.

In [ ]:
# Create the Euler solver.
solver = EulerSolver()

t_beg = time.time()

solver.solve(
    model=model,
    dt=dt,
    n_iters=n_iters,
    period_output=10000,
    dpath_model=dpath_output,
    dpath_morph=dpath_output,
    dpath_pattern=dpath_output,
    verbose=1
)

t_end = time.time()

print("Elapsed time: %f sec." % (t_end - t_beg))

## 8. Inspect Outputs

The solver creates the following directory structure under the output path:
- **`model_1/`**: Contains snapshot images for the first (and only) model, including `pattern_*.png` (raw concentration field) and `morph_*.png` (morph overlay) at each output period.
- **`models/`**: Contains serialized model parameters as JSON files.

In [ ]:
# Sub-directories in the output directory
!ls {dpath_output}

In [ ]:
# Generated images
dpath_images = pjoin(dpath_output, "model_1")
dpath_images

In [ ]:
!ls {dpath_images}

## 9. Final Results

Two types of images are produced at each snapshot:
- **Pattern image**: Shows the raw activator concentration field as a grayscale image, where darker regions indicate higher activator concentration.
- **Morph image**: Overlays the thresholded pattern onto a ladybird wing template, producing a realistic visualization of the predicted color pattern.

In [ ]:
# Show the pattern at last.
img_pattern = Image.open(pjoin(dpath_images, "pattern_500000.png"))
img_pattern

In [ ]:
img_ladybird = Image.open(pjoin(dpath_images, "morph_500000.png"))
img_ladybird

## 10. Model Serialization

The solver saves each model's parameters as a JSON file in the `models/` subdirectory.
This file contains all kinetic parameters, diffusion coefficients, and initial point positions,
enabling full reproducibility of the simulation.

In [ ]:
# Model file
!ls {pjoin(dpath_output, "models")}

In [ ]:
fpath_model = pjoin(dpath_output, "models", "model_1.json")
with open(fpath_model, "rt") as fin:
    model_dict = json.load(fin)
    
model_dict

## 11. Time Evolution Visualization

The `merge_single_timeseries` function collects all snapshot images from a directory and arranges them
into a single grid image, providing an overview of how the pattern evolves over the course of the simulation.
Each cell in the grid is labeled with the iteration number.

In [ ]:
from lpf.visualization import merge_single_timeseries

In [ ]:
# Visualize the temporal evolution of ladybird by merging images.
img_ladybirds = merge_single_timeseries(dpath_input=dpath_images,
                                        n_cols=10,
                                        infile_header="morph",
                                        ratio_resize=0.5,
                                        text_format="t = ",
                                        font_size=10,
                                        text_margin_ratio=.1)
img_ladybirds

In [ ]:
img_ladybirds.save(pjoin(dpath_output, "output_morph.png"))

In [ ]:
img_patterns = merge_single_timeseries(dpath_input=dpath_images,
                                       n_cols=10,
                                       infile_header="pattern",
                                       ratio_resize=0.5,
                                       text_format="t = ",
                                       font_size=10,
                                       text_margin_ratio=.1)
img_patterns

In [ ]:
img_patterns.save(pjoin(dpath_output, "output_pattern.png"))